# Before your start:
- Read the README.md file
- Comment as much as you can and use the resources (README.md file)
- Happy learning!

In [15]:
# import numpy and pandas
import pandas as pd
import numpy as np
from scipy.stats import trim_mean, mode, skew, gaussian_kde, pearsonr, spearmanr, beta

from statsmodels.stats.weightstats import ztest as ztest

from scipy.stats import ttest_ind, norm, t
from scipy.stats import f_oneway
from scipy.stats import sem


# Challenge 1 - Exploring the Data

In this challenge, we will examine all salaries of employees of the City of Chicago. We will start by loading the dataset and examining its contents

In [12]:
# Run this code:
from pathlib import Path
url = Path('..\data\Current_Employee_Names__Salaries__and_Position_Titles.csv')
salaries = pd.read_csv(url)

Examine the `salaries` dataset using the `head` function below.

In [78]:
# Your code here
salaries.head()

,Name,Job Titles,Department,Full or Part-Time,Salary or Hourly,Typical Hours,Annual Salary,Hourly Rate
0,"AARON, JEFFERY M",SERGEANT,POLICE,F,Salary,NaN,101442.0,NaN
1,"AARON, KARINA",POLICE OFFICER (ASSIGNED AS DETECTIVE),POLICE,F,Salary,NaN,94122.0,NaN
2,"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,F,Salary,NaN,101592.0,NaN
3,"ABAD JR, VICENTE M",CIVIL ENGINEER IV,WATER MGMNT,F,Salary,NaN,110064.0,NaN
4,"ABASCAL, REECE E",TRAFFIC CONTROL AIDE-HOURLY,OEMC,P,Hourly,20.0,NaN,19.86


# Challenge 2 - Hypothesis Tests

In this section of the lab, we will test whether the hourly wage of all hourly workers is significantly different from $30/hr. Import the correct one sample test function from scipy and perform the hypothesis test for a 95% two sided confidence interval.

One sided hypothesis test means that:
 
H0: μ <= k

H1: μ > k

In [41]:
from scipy.stats import ttest_1samp
hourly = salaries.loc[salaries["Salary or Hourly"] == "Hourly", "Hourly Rate"].dropna()
hourly = pd.to_numeric(hourly, errors = "coerce")
ttest = ttest_1samp(hourly, popmean = 95, alternative= "two-sided")
print(ttest)
#to check the null hypothesis:
if ttest.pvalue < 0.05:
    print("Reject Null Hypothesis, so mean hourly wage is significantly different after $30/h")
else:
    print("Fail to reject Null Hypothesis, not enough evidence to support the claim.")

TtestResult(statistic=-460.01839948894525, pvalue=0.0, df=8021)
Reject Null Hypothesis, so mean hourly wage is significantly different after $30/h


# Challenge 3 - Constructing Confidence Intervals

While testing our hypothesis is a great way to gather empirical evidence for accepting or rejecting the hypothesis, another way to gather evidence is by creating a confidence interval. A confidence interval gives us information about the true mean of the population. So for a 95% confidence interval, we are 95% sure that the mean of the population is within the confidence interval. 
).

To read more about confidence intervals, click [here](https://en.wikipedia.org/wiki/Confidence_interval).


In the cell below, we will construct a 95% confidence interval for the mean hourly wage of all hourly workers. 

The confidence interval is computed in SciPy using the `t.interval` function. You can read more about this function [here](https://docs.scipy.org/doc/scipy-0.14.0/reference/generated/scipy.stats.t.html).

To compute the confidence interval of the hourly wage, use the 0.95 for the confidence level, number of rows - 1 for degrees of freedom, the mean of the sample for the location parameter and the standard error for the scale. The standard error can be computed using [this](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function in SciPy.

In [55]:
# Your code here
hourly_mean = hourly.mean()
SE = sem(hourly)
CI = t.interval(0.95, df=len(hourly) - 1, loc=hourly_mean, scale=SE)
CI = tuple(round(value, 3) for value in CI)
print(f"SE: {SE:.3f}, CI: {CI}")

SE: 0.135, CI: (32.523, 33.054)


# Challenge 4 - Hypothesis Tests of Proportions

Another type of one sample test is a hypothesis test of proportions. In this test, we examine whether the proportion of a group in our sample is significantly different than a fraction. 

You can read more about one sample proportion tests [here](http://sphweb.bumc.bu.edu/otlt/MPH-Modules/BS/SAS/SAS6-CategoricalData/SAS6-CategoricalData2.html).

In the cell below, use the `proportions_ztest` function from `statsmodels` to perform a hypothesis test that will determine whether the number of hourly workers in the City of Chicago is significantly different from 25% at the 95% confidence level.

In [86]:
# Your code here
from statsmodels.stats.proportion import proportions_ztest
#hypothesis test of whether the number of hourly workers that is significantly different from 25% at 0.95 CL
worker_type = salaries["Salary or Hourly"].dropna()

worker_sum = (worker_type == "Hourly").sum()

print(worker_sum)
nobs = worker_type.size
print(nobs)
#ztest-proportions
ztest_hourly_workers, p_value = proportions_ztest(
                                        count = worker_sum, 
                                         nobs = nobs,
                                         value=0.25, 
                                         alternative="two-sided"
)
print(f"ztest of hourly workers:{ztest_hourly_workers:.3f} , p_value:{p_value:.3f}")
if p_value < 0.05:
    print("Reject H0: the proportion is significantly different from 25%.")
else:
    print("Fail to reject H0: there is not enough evidence that it differs from 25%.")

8022
33183
ztest of hourly workers:-3.510 , p_value:0.000
Reject H0: the proportion is significantly different from 25%.
